# 03 - SFT LoRA Training: Logs, QA, and Multi-task Adapters

This notebook trains task-specific LoRA adapters starting from the base model.

The following adapters are trained:
- Logs classification adapter;
- QA generation adapter;
- Multi-task adapter.

Only the resulting LoRA adapter weights are saved.
The original base model remains unchanged.

In [ ]:
import exp_common as E

RUN_SEEDS = [42, 123, 777]
OVERWRITE = False
MAX_SEQ_LENGTH = 2048
SFT_EPOCHS = 1.0
SFT_MAX_STEPS = -1
BATCH_SIZE = 2
GRAD_ACCUM = 8
LR = 2e-4

TASKS = {
    "logs": (E.PREPARED_DIR / "logs_train_sft.jsonl", E.PREPARED_DIR / "logs_val_sft.jsonl"),
    "qa": (E.PREPARED_DIR / "qa_train_sft.jsonl", E.PREPARED_DIR / "qa_val_sft.jsonl"),
}

for task, (train_path, val_path) in TASKS.items():
    train_rows = E.read_jsonl(train_path)
    val_rows = E.read_jsonl(val_path)
    for seed in RUN_SEEDS:
        out = E.ADAPTERS_DIR / task / f"seed_{seed}"
        adapter = out / "adapter"
        if adapter.exists() and (adapter / "adapter_model.safetensors").exists() and not OVERWRITE:
            print("Exists, skip:", adapter)
            continue
        print("Training", task, "seed", seed)
        E.train_lora_adapter(
            train_rows, val_rows, out, seed=seed, max_seq_length=MAX_SEQ_LENGTH,
            epochs=SFT_EPOCHS, max_steps=SFT_MAX_STEPS,
            per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
            learning_rate=LR, warmup_ratio=0.03,
            lora_r=16, lora_alpha=32, lora_dropout=0.05, overwrite=OVERWRITE,
        )

multi_train = E.read_jsonl(E.PREPARED_DIR / "multitask_train_sft.jsonl")
logs_val = E.read_jsonl(E.PREPARED_DIR / "logs_val_sft.jsonl")
qa_val = E.read_jsonl(E.PREPARED_DIR / "qa_val_sft.jsonl")
multi_val = logs_val + qa_val
for seed in RUN_SEEDS:
    out = E.ADAPTERS_DIR / "multitask" / f"seed_{seed}"
    adapter = out / "adapter"
    if adapter.exists() and (adapter / "adapter_model.safetensors").exists() and not OVERWRITE:
        print("Exists, skip:", adapter)
        continue
    E.train_lora_adapter(
        multi_train, multi_val, out, seed=seed, max_seq_length=MAX_SEQ_LENGTH,
        epochs=SFT_EPOCHS, max_steps=SFT_MAX_STEPS,
        per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR, warmup_ratio=0.03,
        lora_r=16, lora_alpha=32, lora_dropout=0.05, overwrite=OVERWRITE,
    )
print("Base SFT training complete")